# 07 智能摘要压缩

06 做了 FIFO 裁剪，超限就 pop 最老的消息，信息白丢。
07 在裁剪前先调 LLM 摘要，把被裁内容的关键信息压缩保留，注入 system prompt。

**学习点**：上下文管理不仅是"删"，更是"压"——信息密度比信息长度更重要。

In [ ]:
import sys, os, shutil
sys.path.insert(0, '..')
for d in ['agent_memory']:
    if os.path.exists(d): shutil.rmtree(d)

from src.agent_framework import Agent, EmbeddingStore, LLMClient
from src.capabilities import LongTermMemory, PlanManager
from src.capabilities.demo_tools import create_demo_tools

print('一切就绪')

## 对比：有摘要 vs 无摘要

In [ ]:
# 无摘要（跟 06 一样，纯裁剪）
es1 = EmbeddingStore()
ltm1 = LongTermMemory(embedding_store=es1)
agent_no_sum = Agent(
    tools=create_demo_tools(plan_mgr=PlanManager(), ltm=ltm1),
    long_term_memory=ltm1, embedding_store=es1, max_tokens=500,
)
# 注意：没传 llm，所以 Memory 没有 LLM 做摘要，走纯裁剪
# 这里我们手动构造一个对比
print('无摘要 agent 创建完成（走纯裁剪）')

## 有摘要（07 的新能力）

In [ ]:
es2 = EmbeddingStore()
ltm2 = LongTermMemory(embedding_store=es2)
agent_with_sum = Agent(
    tools=create_demo_tools(plan_mgr=PlanManager(), ltm=ltm2),
    long_term_memory=ltm2, embedding_store=es2, max_tokens=500,
)
# Agent 内部自动把 llm 传给 Memory，裁剪时触发摘要
print('有摘要 agent 创建完成（LLM 自动摘要）')

In [ ]:
# 模拟长对话触发摘要
conversation = [
    '你好，我叫程宣赫，今年22岁',
    '我是中国人民大学计算机专业的大三学生',
    '我特别喜欢打篮球，打小前锋位置',
    '我最近在学机器学习，觉得神经网络特别有意思',
]

for msg in conversation:
    reply = agent_with_sum.chat(msg)

print(f'\n累积摘要:\n{agent_with_sum.memory.summary}')
print(f'\n消息数: {len(agent_with_sum.memory._messages)}')
print(f'Token: {agent_with_sum.memory.token_count()}')

In [ ]:
# 即使早期消息被裁，Agent 仍能通过摘要记住关键信息
reply = agent_with_sum.chat('我叫什么名字？我在哪个学校？')
print(f'Agent: {reply[:200]}')

In [ ]:
shutil.rmtree('agent_memory')
print('清理完成')